## SQLite

SQLite is a self-contained, file-based SQL database. SQLite comes bundled with Python and can be used in any Python applications without having to install any additional software.

In [104]:
import sqlite3
from sqlite3 import Error

### Creating a connection to a SQLite database

The sqlite3.connect() function returns a Connection object that we will use to interact with the SQLite database (that ultimately resides in a file). The database (file) will be created automatically if it does not exists.

In [105]:
def create_connection(db_file):
    """ Create a database connection to a SQLite database
    Parameters
    ----------
    db_file: database file
    Returns
    -------
    Connection object or None
    """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        # print(sqlite3.version)
        return conn
    except Error as e:
        print(e)

    return conn

### Creating tables

To create a new table in the SQLite database, first create a Cursor object, by calling the cursor() method of the Connection object, and then pass the CREATE TABLE statement to the execute() method of the Cursor object.

In [106]:
def create_table(conn, create_table_sql):
    """ Create a table from a CREATE TABLE SQL statement
    Parameters
    ----------
    conn: Connection object
    create_table_sql: a CREATE TABLE statement
    """
    try:
        c = conn.cursor()
        c.execute(create_table_sql)
    except Error as e:
        print(e)

### Inserting data

To insert data (rows) into a table, first create a Cursor object, and then execute an INSERT statement. 

To use a variable in a SQL statement, we write a question mark (?) as a placeholder for each argument, and substitute the actual values by providing them as a tuple of values to the cursor’s execute() method.

In SQLite, a column with type INTEGER PRIMARY KEY is an alias for the ROWID, a permanent unique identifier for a row. On an INSERT, if the INTEGER PRIMARY KEY column is not explicitly given a value, then it will be filled automatically with an unused integer, usually one more than the largest ROWID currently in use.

In [107]:
def insert_new_journal(conn, journal):
    """
    Insert a new journal into the journal table
    Parameters
    ----------
    conn: Connection object
    journal: Values to bind to placeholders in sql statement.
    Returns
    -------
    the row id of the inserted row.
    """
    sql = ''' INSERT INTO journal(journal_name, publisher)
              VALUES(?,?) '''
    cur = conn.cursor()
    cur.execute(sql, journal)
    conn.commit()
    return cur.lastrowid

### Querying Data

To query data in the database, first create a Cursor object, then execute a  SELECT statement. After that, call the fetchall() method of the cursor object and loop the cursor to process each row individually.


In [115]:
def select_journals_by_publisher(conn, publisher):
    """
    Query journals by publisher
    Parameters
    ----------
    conn: Connection object
    publisher: Value to bind to placeholder in sql statement.
    """
    cur = conn.cursor()

    # For the question marks style, parameters must be a sequence e.g. a tuple
    # cur.execute("SELECT * FROM journal WHERE publisher=?", (publisher,))

    # Never use string operations to assemble queries, as they are vulnerable to SQL injection attacks
    sql = "SELECT * FROM journal WHERE publisher='%s'" % publisher
    print(sql)
    cur.execute(sql)

    rows = cur.fetchall()

    for row in rows:
        print(row)

In [110]:
database = "./journal_articles.db"

sql_create_journal_table = """CREATE TABLE IF NOT EXISTS journal (
                                   journal_id integer PRIMARY KEY,
                                   journal_name text NOT NULL,
                                   publisher text DEFAULT 'Springer Nature' NOT NULL
                               ); """

In [111]:
# create a database connection
conn = create_connection(database)

In [112]:
# create tables
if conn is not None:
    # create journal table
    create_table(conn, sql_create_journal_table)
else:
    print("Cannot create the database connection.")

In [113]:
# create a new journal
if conn is not None:
    journal = ('Neurocomputing', 'Elsevier');
    journal_id = insert_new_journal(conn, journal)
    journal = ('IEEE Access', 'IEEE');
    journal_id = insert_new_journal(conn, journal)
else:
    print("Cannot create the database connection.")

In [117]:
if conn is not None:
    # Query all rows in the journal table
    # select_all_journals(conn)

    # publisher = 'Elsevier'
    publisher = "' OR TRUE; --"
    select_journals_by_publisher(conn, publisher)

else:
    print("Cannot create the database connection.")

SELECT * FROM journal WHERE publisher='' OR TRUE; --'
(1, 'Neurocomputing', 'Elsevier')
(2, 'IEEE Access', 'IEEE')
(3, 'Neurocomputing', 'Elsevier')
(4, 'IEEE Access', 'IEEE')


In [51]:
if conn:
    conn.close()